# PathBSR Best-Model Prediction Notebook

Purpose: interactively run or inspect the current PathBSR default model on validation or test splits.

- To rerun the default model, set `RUN_MODEL = True` and configure `DATASETS_TO_RUN`, `SPLIT`, and `OUTPUT`.
- The notebook does not run the model automatically on open.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Run this notebook from inside the PathBSR repository")

sys.path.insert(0, str(ROOT / "src"))
RESULTS = ROOT / "results"
DATASETS = ["FB15K-237-10", "FB15K-237-20", "FB15K-237-50", "NELL23K", "WD-singer", "WN18RR"]
print("ROOT =", ROOT)

import subprocess
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
CURRENT_OUTPUT = ROOT / "results/runs/pathbsr_best_model_test.csv"
if CURRENT_OUTPUT.exists():
    final_df = pd.read_csv(CURRENT_OUTPUT)
    cols = ["dataset", "split", "mrr", "hits@1", "hits@3", "hits@10", "num_queries", "build_sec", "eval_sec"]
    display(final_df[[c for c in cols if c in final_df.columns]].sort_values("dataset"))
else:
    print("No current best-model test output yet. Set RUN_MODEL=True below or run scripts/run_pathbsr.py with --output results/runs/pathbsr_best_model_test.csv.")


In [ ]:
# Optional: run the current PathBSR default model.
RUN_MODEL = False
DATASETS_TO_RUN = ["NELL23K"]  # e.g., DATASETS for all six datasets
SPLIT = "valid"
OUTPUT = RESULTS / "runs/notebook_pathbsr_valid.csv"

if RUN_MODEL:
    cmd = [sys.executable, str(ROOT / "scripts/run_pathbsr.py"), "--split", SPLIT, "--output", str(OUTPUT)]
    for dataset in DATASETS_TO_RUN:
        cmd += ["--dataset", dataset]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print("RUN_MODEL=False; set it to True to run PathBSR from this notebook.")

In [ ]:
# Load the notebook run if it exists; otherwise show available precomputed run/detail files.
if OUTPUT.exists():
    run_df = pd.read_csv(OUTPUT)
    display(run_df)
else:
    candidates = sorted((RESULTS / "runs").glob("*.csv"))
    print("No notebook output yet. Available run/detail files:")
    for path in candidates:
        print("-", path.relative_to(ROOT))

In [ ]:
plot_df = final_df.sort_values("dataset")
ax = plot_df.plot(x="dataset", y="mrr", kind="bar", figsize=(8, 3), legend=False)
ax.set_ylabel("MRR")
ax.set_title("PathBSR current default: final test MRR")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()